# Comparação com a Literatura — `GIVPOptimizer.jl`

Experimento **multi-run reproduzível** comparando o GRASP-ILS-VND-PR (`GIVP-full`)
contra baselines em seis funções benchmark clássicas.

| Critério | Valor |
|---|---|
| Linguagem | Julia ≥ 1.9 |
| Pacote | `GIVPOptimizer.jl` v0.8.0 |
| Algoritmos | GIVP-full, GRASP-only |
| Funções | Sphere, Rosenbrock, Rastrigin, Ackley, Griewank, Schwefel |
| Rodadas | 30 por célula (seeds 0–29) |
| Dimensão | 10-D |
| Teste estatístico | Wilcoxon signed-rank (two-sided, α = 0.05) |

> **Pré-requisito:** IJulia instalado e o pacote local `../julia/` acessível.

```julia
# Instalação rápida se necessário:
import Pkg; Pkg.add("IJulia")
```

## 1. Setup e Importações

In [ ]:
import Pkg

# Ativa o pacote local sem re-desenvolver se já ativo
julia_pkg_path = abspath(joinpath(@__DIR__, "..", "..", "julia"))
project_toml = abspath(joinpath(julia_pkg_path, "Project.toml"))
active = isnothing(Base.active_project()) ? "" : abspath(Base.active_project())

if active == project_toml
    println("GIVPOptimizer já ativo no projeto local — pulando Pkg.develop.")
elseif isdir(julia_pkg_path)
    Pkg.develop(path=julia_pkg_path)
else
    Pkg.add("GIVPOptimizer")
end

using GIVPOptimizer
using Printf
using Statistics

println("Julia $(VERSION)")
println("GIVPOptimizer versão: $(GIVPOptimizer.__version__)")
println("Setup concluído ✓")

GIVPOptimizer já ativo no projeto local — pulando Pkg.develop.
Julia 1.12.6
GIVPOptimizer versão: 0.8.0
Setup concluído ✓


## 2. Funções Benchmark

Seis funções clássicas amplamente usadas na literatura de metaheurísticas.
Todas têm mínimo global conhecido em $f^* = 0$ (exceto Schwefel, $f^* = 0$ com $x^* = 418.9829...$).

| Função | Domínio | Características |
|---|---|---|
| Sphere | $[-5.12, 5.12]^n$ | Convexa, unimodal |
| Rosenbrock | $[-5, 10]^n$ | Vale curvo, difícil convergência |
| Rastrigin | $[-5.12, 5.12]^n$ | Multimodal, muitos mínimos locais |
| Ackley | $[-32.768, 32.768]^n$ | Multimodal, platôs exponenciais |
| Griewank | $[-600, 600]^n$ | Muitos mínimos locais regulares |
| Schwefel | $[-500, 500]^n$ | Enganosa, ótimo longe do centro |

In [ ]:
sphere(x) = sum(xi^2 for xi in x)
rosenbrock(x) = sum(100.0 * (x[i+1] - x[i]^2)^2 + (1.0 - x[i])^2 for i in 1:length(x)-1)
rastrigin(x) = 10.0 * length(x) + sum(xi^2 - 10.0 * cos(2π * xi) for xi in x)
ackley(x) = begin
    n = length(x)
    -20.0 * exp(-0.2 * sqrt(sum(xi^2 for xi in x) / n)) - exp(sum(cos(2π * xi) for xi in x) / n) + 20.0 + exp(1.0)
end
griewank(x) = 1.0 + sum(xi^2 for xi in x) / 4000.0 - prod(cos(x[i] / sqrt(Float64(i))) for i in eachindex(x))
schwefel(x) = 418.9829 * length(x) - sum(xi * sin(sqrt(abs(xi))) for xi in x)

const PROBLEMS = Dict(
    "Sphere" => (func=sphere, bounds_fn=n -> [(-5.12, 5.12) for _ in 1:n], optimum=0.0),
    "Rosenbrock" => (func=rosenbrock, bounds_fn=n -> [(-5.0, 10.0) for _ in 1:n], optimum=0.0),
    "Rastrigin" => (func=rastrigin, bounds_fn=n -> [(-5.12, 5.12) for _ in 1:n], optimum=0.0),
    "Ackley" => (func=ackley, bounds_fn=n -> [(-32.768, 32.768) for _ in 1:n], optimum=0.0),
    "Griewank" => (func=griewank, bounds_fn=n -> [(-600.0, 600.0) for _ in 1:n], optimum=0.0),
    "Schwefel" => (func=schwefel, bounds_fn=n -> [(-500.0, 500.0) for _ in 1:n], optimum=0.0),
)

const FUNC_ORDER = ["Sphere", "Rosenbrock", "Rastrigin", "Ackley", "Griewank", "Schwefel"]

println("$(length(PROBLEMS)) funções benchmark definidas ✓")

6 funções benchmark definidas ✓


## 3. Configurações dos Algoritmos

- **GIVP-full**: pipeline completo — GRASP + ILS + VND + Path Relinking + elite pool + cache
- **GRASP-only**: apenas fase de construção GRASP (baseline de Feo & Resende, 1995)

In [3]:
function config_givp_full(; max_iter=100)
    GIVPConfig(
        max_iterations=max_iter,
        alpha=0.12, adaptive_alpha=true, alpha_min=0.08, alpha_max=0.18,
        vnd_iterations=200, ils_iterations=10, perturbation_strength=4,
        use_elite_pool=true, elite_size=7, path_relink_frequency=8,
        use_cache=true, cache_size=10_000,
        early_stop_threshold=80, use_convergence_monitor=true,
    )
end

function config_grasp_only(; max_iter=100)
    GIVPConfig(
        max_iterations=max_iter,
        alpha=0.12, adaptive_alpha=false,
        vnd_iterations=1, ils_iterations=1, perturbation_strength=0,
        use_elite_pool=false, use_convergence_monitor=false,
        use_cache=true, cache_size=10_000,
        early_stop_threshold=max_iter,
    )
end

println("Configurações prontas ✓")

Configurações prontas ✓


## 4. Execução do Experimento

30 rodadas independentes por célula (algoritmo × função), seeds 0–29.  
Cada rodada é independente, com seed fixo para reprodutibilidade.

> Ajuste `N_RUNS` e `DIMS` conforme necessário. Para teste rápido use `N_RUNS=5`.

In [ ]:
const N_RUNS = 30    # rodadas independentes por célula
const DIMS = 10    # dimensionalidade
const MAX_ITER = 100   # máximo de iterações GRASP por rodada

const ALGORITHMS = Dict(
    "GIVP-full" => config_givp_full(max_iter=MAX_ITER),
    "GRASP-only" => config_grasp_only(max_iter=MAX_ITER),
)

# Estrutura de resultado por rodada
struct RunResult
    algorithm::String
    func_name::String
    seed::Int
    fun::Float64
    nit::Int
    nfev::Int
    time_s::Float64
end

function run_all(; verbose=true)
    results = RunResult[]
    for algo_name in ["GIVP-full", "GRASP-only"]
        cfg = ALGORITHMS[algo_name]
        for fn_name in FUNC_ORDER
            prob = PROBLEMS[fn_name]
            bounds = prob.bounds_fn(DIMS)
            verbose && print("  $(rpad(algo_name, 12)) × $(rpad(fn_name, 12)) ")
            for seed in 0:(N_RUNS-1)
                t0 = time()
                res = givp(prob.func, bounds; config=cfg, seed=seed)
                dt = time() - t0
                push!(results, RunResult(algo_name, fn_name, seed,
                    res.fun, res.nit, res.nfev, dt))
            end
            if verbose
                vals = [r.fun for r in results if r.algorithm == algo_name && r.func_name == fn_name]
                @printf("mean=%.4e  best=%.4e  [%d runs]\n", mean(vals), minimum(vals), length(vals))
            end
        end
    end
    return results
end

println("Iniciando experimento: $(length(ALGORITHMS)) algoritmos × $(length(PROBLEMS)) funções × $N_RUNS rodadas...")
results = run_all(verbose=true)
println("\n✓ Experimento concluído — $(length(results)) rodadas totais")

Iniciando experimento: 2 algoritmos × 6 funções × 30 rodadas...
  GIVP-full    × Sphere       mean=2.0692e-04  best=6.1780e-05  [30 runs]
  GIVP-full    × Rosenbrock   mean=4.3611e-01  best=9.6420e-03  [30 runs]
  GIVP-full    × Rastrigin    mean=9.8794e-01  best=8.9753e-03  [30 runs]
  GIVP-full    × Ackley       mean=1.3495e-01  best=9.3450e-02  [30 runs]
  GIVP-full    × Griewank     mean=1.7085e-01  best=9.7227e-02  [30 runs]
  GIVP-full    × Schwefel     mean=4.9916e+01  best=1.2843e-04  [30 runs]
  GRASP-only   × Sphere       mean=2.5144e+00  best=1.1490e+00  [30 runs]
  GRASP-only   × Rosenbrock   mean=1.1314e+04  best=2.2161e+03  [30 runs]
  GRASP-only   × Rastrigin    mean=3.9875e+01  best=1.8725e+01  [30 runs]
  GRASP-only   × Ackley       mean=1.0992e+01  best=9.0002e+00  [30 runs]
  GRASP-only   × Griewank     mean=9.5772e+00  best=4.9104e+00  [30 runs]
  GRASP-only   × Schwefel     mean=1.9143e+03  best=1.5491e+03  [30 runs]

✓ Experimento concluído — 360 rodadas totais


## 5. Estatísticas Descritivas

Calculadas sobre as 30 rodadas: média, desvio padrão, melhor valor, mediana e pior valor.

In [ ]:
function summary_table(results::Vector{RunResult})
    rows = []
    for fn in FUNC_ORDER, algo in ["GIVP-full", "GRASP-only"]
        vals = [r.fun for r in results if r.func_name == fn && r.algorithm == algo]
        isempty(vals) && continue
        push!(rows, (
            Function=fn,
            Algorithm=algo,
            N=length(vals),
            Mean=mean(vals),
            Std=std(vals),
            Best=minimum(vals),
            Median=median(vals),
            Worst=maximum(vals),
        ))
    end
    return rows
end

smry = summary_table(results)

# Pretty-print console table (rpad/lpad avoids @printf compile-time format limitations)
C1, C2, CN = 12, 16, 14
sep = "  " * "─"^(C1 + 2 + C2 + CN * 4)
println()
println("  " * rpad("Function", C1) * "  " * rpad("Algorithm", C2) *
        lpad("Mean", CN) * lpad("Std", CN) * lpad("Best", CN) * lpad("Median", CN))
println(sep)
for r in smry
    println("  " * rpad(r.Function, C1) * "  " * rpad(r.Algorithm, C2) *
            lpad(@sprintf("%.4e", r.Mean), CN) *
            lpad(@sprintf("%.4e", r.Std), CN) *
            lpad(@sprintf("%.4e", r.Best), CN) *
            lpad(@sprintf("%.4e", r.Median), CN))
end


  Function      Algorithm                 Mean           Std          Best        Median
  ──────────────────────────────────────────────────────────────────────────────────────
  Sphere        GIVP-full           2.0692e-04    6.3134e-05    6.1780e-05    2.1925e-04
  Sphere        GRASP-only          2.5144e+00    5.6531e-01    1.1490e+00    2.6281e+00
  Rosenbrock    GIVP-full           4.3611e-01    3.1966e-01    9.6420e-03    3.9216e-01
  Rosenbrock    GRASP-only          1.1314e+04    5.7702e+03    2.2161e+03    1.0644e+04
  Rastrigin     GIVP-full           9.8794e-01    6.0726e-01    8.9753e-03    1.0140e+00
  Rastrigin     GRASP-only          3.9875e+01    7.4324e+00    1.8725e+01    4.1867e+01
  Ackley        GIVP-full           1.3495e-01    2.6316e-02    9.3450e-02    1.3192e-01
  Ackley        GRASP-only          1.0992e+01    7.9834e-01    9.0002e+00    1.1172e+01
  Griewank      GIVP-full           1.7085e-01    3.7264e-02    9.7227e-02    1.7058e-01
  Griewank      GRAS

## 6. Teste de Wilcoxon Signed-Rank

Teste não-paramétrico de significância estatística comparando cada algoritmo
contra o **GIVP-full** (referência).

- **W**: estatística do teste
- **p**: p-valor (two-sided)
- **r**: correlação rank-biserial (tamanho do efeito)
- **★**: p < 0.05 — diferença estatisticamente significativa

In [ ]:
# ── Wilcoxon signed-rank (normal approximation with continuity correction) ──

function _normal_cdf(x::Float64)
    t = 1.0 / (1.0 + 0.3275911 * abs(x))
    y = t * (0.254829592 + t * (-0.284496736 + t * (1.421413741 + t * (-1.453152027 + t * 1.061405429))))
    p = 1.0 - y * exp(-x * x)
    x < 0 ? 1.0 - p : p
end

function wilcoxon_test(a::Vector{Float64}, b::Vector{Float64})
    d = filter(!iszero, a .- b)
    n = length(d)
    n < 2 && return (0.0, 1.0, 0.0)
    abs_d = abs.(d)
    ord = sortperm(abs_d)
    ranks = zeros(Float64, n)
    i = 1
    while i <= n
        j = i
        while j < n && abs_d[ord[j+1]] ≈ abs_d[ord[i]]
            j += 1
        end
        avg = (i + j) / 2.0
        for k in i:j
            ranks[ord[k]] = avg
        end
        i = j + 1
    end
    W_plus = sum(ranks[k] for k in 1:n if d[k] > 0; init=0.0)
    W = min(W_plus, sum(ranks) - W_plus)
    r_eff = 1.0 - 2.0 * W / (n * (n + 1) / 2.0)
    mu = n * (n + 1) / 4.0
    sigma = sqrt(n * (n + 1) * (2n + 1) / 24.0)
    z = (W - mu + 0.5) / sigma
    pval = 2.0 * _normal_cdf(-abs(z))
    return W, pval, r_eff
end

# ── Run tests ─────────────────────────────────────────────────────────────────

reference = "GIVP-full"
challengers = ["GRASP-only"]
α = 0.05

C1, C2 = 12, 16
println("Wilcoxon signed-rank test (two-sided, α=$α, reference=$reference)\n")
println("  " * rpad("Function", C1) * "  " * rpad("Challenger", C2) *
        lpad("W", 8) * lpad("p-value", 10) * lpad("r", 8) * lpad("Sig", 5))
println("  " * "─"^(C1 + 2 + C2 + 8 + 10 + 8 + 5))

wilcoxon_results = []
for fn in FUNC_ORDER, chal in challengers
    ref_seeds = [r.seed for r in results if r.func_name == fn && r.algorithm == reference]
    chal_seeds = [r.seed for r in results if r.func_name == fn && r.algorithm == chal]
    common = intersect(ref_seeds, chal_seeds)
    length(common) < 2 && continue

    a = [r.fun for s in common for r in results if r.func_name == fn && r.algorithm == reference && r.seed == s]
    b = [r.fun for s in common for r in results if r.func_name == fn && r.algorithm == chal && r.seed == s]
    W, pval, r_eff = wilcoxon_test(a, b)
    sig = pval < α ? "★" : "—"
    push!(wilcoxon_results, (fn, chal, W, pval, r_eff, pval < α))
    println("  " * rpad(fn, C1) * "  " * rpad(chal, C2) *
            lpad(@sprintf("%.1f", W), 8) *
            lpad(@sprintf("%.4f", pval), 10) *
            lpad(@sprintf("%.3f", r_eff), 8) *
            lpad(sig, 5))
end
println("\n★ = p < $α  (significantly different from $reference)")

Wilcoxon signed-rank test (two-sided, α=0.05, reference=GIVP-full)

  Function      Challenger             W   p-value       r  Sig
  ─────────────────────────────────────────────────────────────
  Sphere        GRASP-only           0.0    0.0000   1.000    ★
  Rosenbrock    GRASP-only           0.0    0.0000   1.000    ★
  Rastrigin     GRASP-only           0.0    0.0000   1.000    ★
  Ackley        GRASP-only           0.0    0.0000   1.000    ★
  Griewank      GRASP-only           0.0    0.0000   1.000    ★
  Schwefel      GRASP-only           0.0    0.0000   1.000    ★

★ = p < 0.05  (significantly different from GIVP-full)


## 7. Tabela Markdown (pronta para README / artigo)

Gerada automaticamente no formato compatível com GitHub Markdown e Overleaf.

In [ ]:
md_lines = String[]
push!(md_lines, "<!-- GIVPOptimizer.jl v$(GIVPOptimizer.__version__) | n=$DIMS | $N_RUNS runs -->")
push!(md_lines, "")

pval_idx = Dict((r[1], r[2]) => r[4] for r in wilcoxon_results)
sig_idx = Dict((r[1], r[2]) => r[6] for r in wilcoxon_results)

for fn in FUNC_ORDER
    push!(md_lines, "### $fn")
    push!(md_lines, "")
    push!(md_lines, "| Algorithm | Mean ± Std | Best | Median | Worst | p-value | Sig |")
    push!(md_lines, "|-----------|------------|------|--------|-------|---------|-----|")
    for r in filter(x -> x.Function == fn, smry)
        pval_str = haskey(pval_idx, (fn, r.Algorithm)) ?
                   @sprintf("%.4f", pval_idx[(fn, r.Algorithm)]) : "*(ref)*"
        sig_str = get(sig_idx, (fn, r.Algorithm), false) ? "★" : "—"
        push!(md_lines, "| $(r.Algorithm) " *
                        "| $(@sprintf("%.4e", r.Mean)) ± $(@sprintf("%.4e", r.Std)) " *
                        "| $(@sprintf("%.4e", r.Best)) " *
                        "| $(@sprintf("%.4e", r.Median)) " *
                        "| $(@sprintf("%.4e", r.Worst)) " *
                        "| $pval_str | $sig_str |")
    end
    push!(md_lines, "")
end
push!(md_lines, "★ p < 0.05 (Wilcoxon signed-rank, two-sided) vs. *$reference*.")

md_table = join(md_lines, "\n")
println(md_table)

<!-- GIVPOptimizer.jl v0.8.0 | n=10 | 30 runs -->

### Sphere

| Algorithm | Mean ± Std | Best | Median | Worst | p-value | Sig |
|-----------|------------|------|--------|-------|---------|-----|
| GIVP-full | 2.0692e-04 ± 6.3134e-05 | 6.1780e-05 | 2.1925e-04 | 3.2559e-04 | *(ref)* | — |
| GRASP-only | 2.5144e+00 ± 5.6531e-01 | 1.1490e+00 | 2.6281e+00 | 3.4721e+00 | 0.0000 | ★ |

### Rosenbrock

| Algorithm | Mean ± Std | Best | Median | Worst | p-value | Sig |
|-----------|------------|------|--------|-------|---------|-----|
| GIVP-full | 4.3611e-01 ± 3.1966e-01 | 9.6420e-03 | 3.9216e-01 | 1.3139e+00 | *(ref)* | — |
| GRASP-only | 1.1314e+04 ± 5.7702e+03 | 2.2161e+03 | 1.0644e+04 | 2.3499e+04 | 0.0000 | ★ |

### Rastrigin

| Algorithm | Mean ± Std | Best | Median | Worst | p-value | Sig |
|-----------|------------|------|--------|-------|---------|-----|
| GIVP-full | 9.8794e-01 ± 6.0726e-01 | 8.9753e-03 | 1.0140e+00 | 2.0288e+00 | *(ref)* | — |
| GRASP-only | 3.9875e+01 ± 7.4324e+0

## 8. Tabela LaTeX (booktabs — pronta para SBPO / BRACIS)

Requer `\usepackage{booktabs}` e `\usepackage{multirow}` no preâmbulo.

In [ ]:
function latex_esc(s)
    replace(replace(s, "_" => "\\_"), "&" => "\\&")
end

tex_lines = String[]
push!(tex_lines, "% GIVPOptimizer.jl v$(GIVPOptimizer.__version__) | n=$DIMS | $N_RUNS runs")
push!(tex_lines, "")
push!(tex_lines, "\\begin{table}[htb]")
push!(tex_lines, "\\centering")
push!(tex_lines, "\\caption{Comparação de algoritmos em funções benchmark clássicas " *
                 "(\$n=$DIMS\$, $N_RUNS rodadas). Média \$\\pm\$ desvio padrão. " *
                 "\$^\\star\$ indica \$p < 0.05\$ (Wilcoxon, two-sided) vs.\\ \\texttt{$(latex_esc(reference))}.}}")
push!(tex_lines, "\\label{tab:givp_julia_benchmark}")
push!(tex_lines, "\\begin{tabular}{llrrrr}")
push!(tex_lines, "\\toprule")
push!(tex_lines, "Função & Algoritmo & Média & Desvio & Melhor & Mediana \\\\")
push!(tex_lines, "\\midrule")

for fn in FUNC_ORDER
    fn_rows = filter(x -> x.Function == fn, smry)
    for (i, r) in enumerate(fn_rows)
        fn_cell = i == 1 ? "\\multirow{$(length(fn_rows))}{*}{$fn}" : ""
        sig = get(sig_idx, (fn, r.Algorithm), false) ? "\$^\\star\$" : ""
        push!(tex_lines, "  $fn_cell & $(latex_esc(r.Algorithm))$sig " *
                         "& \$$(@sprintf("%.4e", r.Mean))\$ " *
                         "& \$$(@sprintf("%.4e", r.Std))\$ " *
                         "& \$$(@sprintf("%.4e", r.Best))\$ " *
                         "& \$$(@sprintf("%.4e", r.Median))\$ \\\\")
    end
    push!(tex_lines, "\\midrule")
end
tex_lines[end] = "\\bottomrule"
push!(tex_lines, "\\end{tabular}")
push!(tex_lines, "\\end{table}")

println(join(tex_lines, "\n"))

% GIVPOptimizer.jl v0.8.0 | n=10 | 30 runs

\begin{table}[htb]
\centering
\caption{Comparação de algoritmos em funções benchmark clássicas ($n=10$, 30 rodadas). Média $\pm$ desvio padrão. $^\star$ indica $p < 0.05$ (Wilcoxon, two-sided) vs.\ \texttt{GIVP-full}.}}
\label{tab:givp_julia_benchmark}
\begin{tabular}{llrrrr}
\toprule
Função & Algoritmo & Média & Desvio & Melhor & Mediana \\
\midrule
  \multirow{2}{*}{Sphere} & GIVP-full & $2.0692e-04$ & $6.3134e-05$ & $6.1780e-05$ & $2.1925e-04$ \\
   & GRASP-only$^\star$ & $2.5144e+00$ & $5.6531e-01$ & $1.1490e+00$ & $2.6281e+00$ \\
\midrule
  \multirow{2}{*}{Rosenbrock} & GIVP-full & $4.3611e-01$ & $3.1966e-01$ & $9.6420e-03$ & $3.9216e-01$ \\
   & GRASP-only$^\star$ & $1.1314e+04$ & $5.7702e+03$ & $2.2161e+03$ & $1.0644e+04$ \\
\midrule
  \multirow{2}{*}{Rastrigin} & GIVP-full & $9.8794e-01$ & $6.0726e-01$ & $8.9753e-03$ & $1.0140e+00$ \\
   & GRASP-only$^\star$ & $3.9875e+01$ & $7.4324e+00$ & $1.8725e+01$ & $4.1867e+01$ \\
\midrule
  \mu

## 9. Exportar Resultados para JSON

Salva os resultados em formato compatível com `generate_report.jl` para uso posterior.

In [ ]:
using JSON
using Dates

output_path = joinpath(@__DIR__, "results_notebook_julia.json")

records_json = [Dict(
    "algorithm" => r.algorithm,
    "function" => r.func_name,
    "seed" => r.seed,
    "fun" => r.fun,
    "nit" => r.nit,
    "nfev" => r.nfev,
    "time_s" => r.time_s,
) for r in results]

summary_json = [Dict(
    "function" => r.Function,
    "algorithm" => r.Algorithm,
    "n_runs" => r.N,
    "mean" => r.Mean,
    "std" => r.Std,
    "best" => r.Best,
    "median" => r.Median,
    "worst" => r.Worst,
    "nfev_mean" => mean(Float64[rr.nfev for rr in results if rr.func_name == r.Function && rr.algorithm == r.Algorithm]),
) for r in smry]

data = Dict(
    "metadata" => Dict(
        "julia_version" => string(VERSION),
        "givp_version" => GIVPOptimizer.__version__,
        "dims" => DIMS,
        "n_runs" => N_RUNS,
        "algorithms" => collect(keys(ALGORITHMS)),
        "functions" => FUNC_ORDER,
        "generated_at" => string(now()),
    ),
    "summary" => summary_json,
    "records" => records_json,
)

open(output_path, "w") do io
    JSON.print(io, data, 2)
end

println("Resultados salvos em: $output_path")
println("Use generate_report.jl para relatórios Markdown/LaTeX:")
println("  julia --project=julia julia/benchmarks/generate_report.jl --input $output_path")

Resultados salvos em: d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr\Notebooks\Julia\results_notebook_julia.json
Use generate_report.jl para relatórios Markdown/LaTeX:
  julia --project=julia julia/benchmarks/generate_report.jl --input d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr\Notebooks\Julia\results_notebook_julia.json


## 10. Interpretação dos Resultados

### Como ler a tabela
- **Média ± Std**: qualidade média e consistência nas 30 rodadas (menor = melhor)
- **Melhor**: solução ótima encontrada em qualquer das 30 rodadas
- **p-value**: probabilidade de os resultados serem iguais por acaso
- **★**: GRASP-only é significativamente diferente do GIVP-full (p < 0.05)

### Hipótese testada
$H_0$: GIVP-full e GRASP-only produzem distribuições equivalentes de valores objetivos.  
$H_1$: As distribuições são diferentes.

Um ★ indica que o GIVP-full é estatisticamente superior ao GRASP-only na função em questão —
evidência científica publicável para SBPO / BRACIS.

### Reproduzindo este experimento
```julia
# Via script de linha de comando (sem notebook):
julia --project=julia julia/benchmarks/run_literature_comparison.jl \\
    --n-runs 30 --dims 10 --output results.json --verbose

julia --project=julia julia/benchmarks/generate_report.jl \\
    --input results.json --format both
```